# Three-PC reviewed-Gold backbone comparison

This notebook uses one shared scientific contract and three independent launch cells. Run cells 1-6 on every PC, then run only the assigned PC cell: PC 1 = Qwen2-VL-2B, PC 2 = Qwen2.5-VL-7B, PC 3 = InternVL3.5-8B-HF. Each model writes to its own persistent directory, saves an exact-batch `last.ckpt`, keeps all completed epoch metrics, selects checkpoints only on the 7,861 original-Gold validation rows, reports the 194 supplement rows separately, and never opens the locked test split.

A new backbone automatically runs a 16-row implementation compatibility gate and a controlled 5,000-row mini before full training. These gates test the new processor/encoder path; they do not re-review the finalized dataset. If either gate fails, the expensive full run is blocked. Qwen2-VL-2B reuses its accepted v2.8 mini evidence.

In [ ]:
# 1. Shared paths and immutable experiment settings.
from pathlib import Path
import os

REPO_ROOT = Path('/home/aiub/kiyas/webagent').resolve()
DATA_HOME = Path('/home/aiub/kiyas/webagent_full/data').resolve()
ORIGINAL_SEARCH_ROOT = DATA_HOME / 'original'
SUPPLEMENT_SEARCH_ROOT = DATA_HOME / 'supplement'
WORKSPACE_ROOT = Path('/home/aiub/kiyas/webagent_comparison').resolve()
SEED = 42
FULL_EPOCHS = 10
MINI_EPOCHS = 5
NUM_WORKERS = 8
SELECTION_RULE = 'all_gates_then_outcome_mcc'
TWO_PERSON_NO_CHANGE_REVIEW_CONFIRMED = True
REVIEWER_COUNT = 2

assert REPO_ROOT.joinpath('.git').is_dir(), REPO_ROOT
assert ORIGINAL_SEARCH_ROOT.is_dir(), ORIGINAL_SEARCH_ROOT
assert SUPPLEMENT_SEARCH_ROOT.is_dir(), SUPPLEMENT_SEARCH_ROOT
assert WORKSPACE_ROOT != Path('/tmp')
assert TWO_PERSON_NO_CHANGE_REVIEW_CONFIRMED and REVIEWER_COUNT == 2
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(WORKSPACE_ROOT / 'hf_cache')


In [ ]:
# 2. Verify the existing DGX runtime; do not replace the CUDA build of torch.
import importlib.metadata as metadata
import json, platform, subprocess, sys
from packaging.version import Version
import torch

assert torch.cuda.is_available(), 'A CUDA GPU is required.'
assert torch.cuda.is_bf16_supported(), 'The 7B/8B profiles require bf16.'
transformers_version = Version(metadata.version('transformers'))
assert Version('4.57.6') <= transformers_version < Version('5.0.0'), transformers_version
for package in ('peft', 'bitsandbytes', 'accelerate', 'scikit-learn', 'pandas', 'pyyaml'):
    metadata.version(package)
environment = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'transformers': str(transformers_version),
    'gpu': torch.cuda.get_device_name(0),
    'gpu_total_memory_gb': torch.cuda.get_device_properties(0).total_memory / 1e9,
}
assert environment['gpu_total_memory_gb'] >= 120, environment
print(json.dumps(environment, indent=2))


In [ ]:
# 3. Freeze the current checkout and import this repository. Pull before opening the notebook, never during resume.
SOURCE_ROOT = REPO_ROOT / 'src'
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
os.environ['PYTHONPATH'] = str(SOURCE_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
os.chdir(REPO_ROOT)
import web_agent
assert Path(web_agent.__file__).resolve().is_relative_to(SOURCE_ROOT)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
environment['git_commit'] = CODE_COMMIT
environment_path = WORKSPACE_ROOT / 'environment.json'
environment_path.write_text(json.dumps(environment, indent=2), encoding='utf-8')
print('Frozen commit:', CODE_COMMIT)


In [ ]:
# 4. Resolve the already-downloaded original and supplement data without opening split_test.json.
from web_agent.data.recovery_supplement import resolve_supplement_root

def find_original_root(search_root: Path) -> Path:
    candidates = sorted({
        path.parent.resolve() for path in search_root.rglob('split_train.json')
        if path.parent.joinpath('split_val.json').is_file() and path.parent.joinpath('images').is_dir()
    })
    assert len(candidates) == 1, f'Expected one extracted original root; found {candidates}'
    return candidates[0]

ORIGINAL_ROOT = find_original_root(ORIGINAL_SEARCH_ROOT)
SUPPLEMENT_ROOT = resolve_supplement_root(SUPPLEMENT_SEARCH_ROOT)
locked_test_paths = sorted(ORIGINAL_SEARCH_ROOT.rglob('split_test.json'))
print('Locked test file may be present but is not opened:', locked_test_paths)
print('Original:', ORIGINAL_ROOT)
print('Supplement:', SUPPLEMENT_ROOT)


In [ ]:
# 5. Cache and verify source gates shared by all three candidates.
PREFLIGHT_ROOT = WORKSPACE_ROOT / 'preflight'
PREFLIGHT_ROOT.mkdir(parents=True, exist_ok=True)
SUPPLEMENT_REPORT = PREFLIGHT_ROOT / 'supplement_validation.json'
MULTISOURCE_REPORT = PREFLIGHT_ROOT / 'multisource_audit.json'
commands = [
    [sys.executable, 'scripts/validate_retry_abort_supplement.py', '--supplement-root', str(SUPPLEMENT_ROOT), '--report', str(SUPPLEMENT_REPORT)],
    [sys.executable, 'scripts/audit_retry_abort_multisource.py', '--original-root', str(ORIGINAL_ROOT), '--supplement-root', str(SUPPLEMENT_ROOT), '--report', str(MULTISOURCE_REPORT)],
]
for command, report_path in zip(commands, (SUPPLEMENT_REPORT, MULTISOURCE_REPORT)):
    existing = json.loads(report_path.read_text(encoding='utf-8')) if report_path.is_file() else None
    if not existing or existing.get('status') != 'PASS' or existing.get('test_rows_read') != 0:
        subprocess.run(command, check=True)
    gate = json.loads(report_path.read_text(encoding='utf-8'))
    assert gate['status'] == 'PASS' and gate['test_rows_read'] == 0, report_path
multisource = json.loads(MULTISOURCE_REPORT.read_text(encoding='utf-8'))
assert multisource['primary_training_rows'] == 24_107
assert multisource['primary_validation_rows'] == 7_861
assert multisource['supplement_validation_rows_available_separately'] == 194
print('COMMON DATA CONTRACT PASSED: 24,107 train / 7,861 primary val / 194 supplement val / 0 test')


In [ ]:
# 6. Shared launcher. phase='auto' means compatibility -> required mini -> full/resume.
def run_candidate(model_id: str, *, phase: str = 'auto') -> None:
    assert phase in {'auto', 'mini', 'full'}
    command = [
        sys.executable, 'scripts/run_comparison_candidate.py',
        '--model-id', model_id, '--phase', phase,
        '--data-root', str(ORIGINAL_ROOT),
        '--supplement-root', str(SUPPLEMENT_ROOT),
        '--workspace-root', str(WORKSPACE_ROOT),
        '--seed', str(SEED), '--epochs', str(FULL_EPOCHS),
        '--mini-epochs', str(MINI_EPOCHS), '--num-workers', str(NUM_WORKERS),
    ]
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, check=True)


In [ ]:
# 7. PC 1 only — existing reference backbone.
run_candidate('qwen2vl_2b_gold_v2_8_dgx', phase='auto')


In [ ]:
# 8. PC 2 only — Qwen2.5-VL-7B candidate.
run_candidate('qwen25vl_7b_gold_v2_8_dgx', phase='auto')


In [ ]:
# 9. PC 3 only — InternVL3.5-8B-HF candidate.
run_candidate('internvl35_8b_gold_v2_8_dgx', phase='auto')


In [ ]:
# 10. After copying all three complete model directories onto one PC, rank validation results.
COMPARISON_ROOT = WORKSPACE_ROOT / 'outputs' / 'model_comparison'
DECISION_ROOT = WORKSPACE_ROOT / 'outputs' / 'comparison_decision'
subprocess.run([
    sys.executable, 'scripts/compare_full_models.py',
    '--comparison-root', str(COMPARISON_ROOT), '--seed', str(SEED),
    '--csv', str(DECISION_ROOT / 'three_model_validation_comparison.csv'),
    '--json', str(DECISION_ROOT / 'three_model_selection.json'),
], check=True)


## Final decision boundary

The comparison accepts only full reports that pass every registered quality gate. It ranks eligible candidates by original-validation outcome MCC, then uses recovery-outcome MCC, action macro-F1, lower outcome ECE, and model ID only as deterministic tie-breakers. The locked test remains unopened. After selecting the winner, run that one backbone with seeds 43 and 44; only then evaluate the frozen three-seed model on the locked test splits for the paper.